# Create Your Own Processing Pipeline

In this tutorial, you will learn how to create a custom data processing pipeline. Specifically, this example will:
1. Perform data augmentation on a dataset version.
2. Upload the augmented dataset version to Picsellia for further use.

To begin, you need to define a **processing context** for the pipeline. This context provides the necessary configuration and parameters required for the pipeline to execute.

To test the pipeline locally, you can use the `create_local_processing_context` function. This function simulates a Picsellia processing environment and allows you to debug the pipeline without connecting to Picsellia's infrastructure.

For this example, we will define the following parameters:
- **`augmentation_probability`**: The probability of applying augmentations to the images.
- **`datalake`**: The target Picsellia datalake where the augmented dataset will be stored.
- **`data_tag`**: A tag that identifies the augmented dataset in the Picsellia datalake.

Here’s how to define these parameters:

In [1]:
parameters = {
    "augmentation_probability": 0.5,
    "datalake": "default",
    "data_tag": "augmented_data",
}


Next, you need to create the local processing context using the `create_local_processing_context` function. This function initializes a local context with the specified parameters, simulating a Picsellia processing job.

### Required Inputs:
- **`api_token`**: Your Picsellia API token.
- **`organization_id`**: Your Picsellia organization ID.
- **`job_id`**: A unique name for your processing job. A directory with this name will be created locally.
- **`job_type`**: The type of processing job. For this pipeline, the job type is `ProcessingType.DATASET_VERSION_CREATION`.
- **`input_dataset_version_id`**: The ID of the dataset version to process.
- **`output_dataset_version_id`**: The ID of an empty dataset version where the augmented data will be saved.
- **`processing_parameters`**: The dictionary of processing parameters defined earlier.

Here’s an example:

In [2]:
from src.models.utils.local_context import create_local_processing_context
from picsellia.types.enums import ProcessingType


local_processing_context = create_local_processing_context(api_token="500ccf0e5f12197766a4adb4ac6e8838ccf30268", organization_id="9669529a-4ffd-4a53-8a6d-aae8584a8ae1", job_id="test_augmentations", job_type=ProcessingType.DATASET_VERSION_CREATION, input_dataset_version_id="019301e7-bd75-778c-abf1-16691b58aa2a", output_dataset_version_id="01944673-2031-7af0-94c8-8f353289881e", processing_parameters={"augmentation_probability": 0.5, "datalake": "default", "data_tag": "augmented_data",})

You are using an outdated version of the picsellia package (6.18.2)
Please consider upgrading to 6.18.3 with pip install picsellia --upgrade
Hi SoniaGrh, welcome back. 🥑
Workspace: SoniaGrh's organization.


Now that the processing context is created, let’s explore its key attributes.

The **`processing_parameters`** attribute contains the parameters you defined earlier. These parameters are easily accessible using dot notation. For instance:

- `local_processing_context.processing_parameters.augmentation_probability` will return the value of the `augmentation_probability` parameter.


In [3]:
local_processing_context.processing_parameters.augmentation_probability

0.5

If you want to deploy the pipeline on the Picsellia platform instead of running it locally, you can use the `create_picsellia_processing_context` function.

This function creates a **PicselliaProcessingContext**, which interacts directly with Picsellia’s backend to execute the pipeline on the platform.

Here’s how you would define it (commented out in this example since the local context is being used):


In [4]:
# from src.models.utils.picsellia_context import create_picsellia_processing_context
#
# picsellia_processing_context = create_picsellia_processing_context(parameters)

## Step 1: Fetching Datasets

The first step in the pipeline is to fetch the datasets required for processing. This is handled by the `get_processing_dataset_collection` function.

### Key Points:
- **Input Dataset**: The original dataset to be processed.
- **Output Dataset**: An empty dataset where the processed data will be saved.
- Both datasets are automatically downloaded, and their annotations are pre-loaded.

To encapsulate this logic in a pipeline, we use the `@pipeline` decorator with the local context created earlier.

In [5]:
from src.steps.data_extraction.processing.processing_data_extractor import get_processing_dataset_collection
from src import pipeline

@pipeline(context=local_processing_context, log_folder_path="logs/", remove_logs_on_completion=False)
def augmentations_pipeline():
    dataset_collection = get_processing_dataset_collection()
    return dataset_collection

Once the pipeline is defined, you can execute it to fetch the datasets. The result is a **dataset collection** containing:
- **`dataset_collection["input"]`**: Represents the input dataset.
- **`dataset_collection["output"]`**: Represents the output dataset.

In [6]:
dataset_collection = augmentations_pipeline()

Log folder created at /home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/examples/processing/augmentation/logs
Pipeline augmentations_pipeline is starting with the following context:
| parameters            | values                                                             |
|-----------------------|--------------------------------------------------------------------|
| processing_parameters | create_local_processing_parameters.<locals>.ProcessingParameters() |

| context_parameters        | values                                  |
|---------------------------|-----------------------------------------|
| host                      | https://app.picsellia.com               |
| organization_id           | 9669529a-4ffd-4a53-8a6d-aae8584a8ae1    |
| job_type                  | ProcessingType.DATASET_VERSION_CREATION |
| input_dataset_version_id  | 019301e7-bd75-778c-abf1-16691b58aa2a    |
| output_dataset_version_id | 01944673-2031-7af0-94c8-8f353289881e    |
| mo

100%|███████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 12.59it/s]

20 assets downloaded (over 20) in directory /home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/examples/processing/augmentation/test_augmentations/images/input


No assets found in the dataset version, skipping asset download.
No assets found in the dataset version, skipping asset listing.


No batches were successfully downloaded.
(1/1) get_processing_dataset_collection execution time: 2.669 seconds


In [7]:
dataset_collection["input"]

In [8]:
dataset_collection["output"]

Now that we have retrieved the datasets, we have two components:
- **input_dataset**: Contains the original data to be processed.
- **output_dataset**: Will store the results of the processing.

Next, let’s explore the key attributes of these datasets.

### Input Dataset Attributes

The input dataset, accessible through input_dataset, includes the following attributes:

- `images_dir`: Directory containing the already downloaded images.
- `coco_file_path`: Path to the COCO annotation file for the dataset version.
- `coco_data`: COCO dictionary loaded from the annotation file.
- `dataset_version`: Object representing the dataset version in Picsellia.

In [9]:
dataset_collection["input"].images_dir

'/home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/examples/processing/augmentation/test_augmentations/images/input'

In [10]:
dataset_collection["input"].coco_file_path

'/home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/examples/processing/augmentation/test_augmentations/annotations/input/merged_coco_annotations.json'

In [11]:
dataset_collection["input"].coco_data["images"][0]

{'id': 0,
 'file_name': '019301e7-be6a-76cd-99de-625f6c6cd78e.jpg',
 'width': 929,
 'height': 580}

In [12]:
dataset_collection["input"].dataset_version

Version 'small-sample' of dataset Identity  (id: 019301e7-bd75-778c-abf1-16691b58aa2a)

### Output Dataset Attributes

The output_dataset shares the same attributes as the input_dataset, with the following key differences:

- `images_dir`: Initially empty; this is where you will save the augmented images.
- `coco_file_path`: Set to None by default; you will need to provide a path to the annotations for the augmented dataset.
- `coco_data`: Set to None by default; it will need to be populated with COCO-format annotations.
- `dataset_version`: A Picsellia dataset version object that has already been created.


In [13]:
dataset_collection["output"].images_dir

'/home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/examples/processing/augmentation/test_augmentations/images/output'

In [14]:
dataset_collection["output"].annotations_dir

'/home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/examples/processing/augmentation/test_augmentations/annotations/output'

In [15]:
dataset_collection["output"].coco_file_path

In [16]:
dataset_collection["output"].coco_data

In [17]:
dataset_collection["output"].dataset_version

Version 'augmented_2' of dataset Identity  (id: 01944673-2031-7af0-94c8-8f353289881e)

Let’s define an example function for applying image augmentations. In this case, we will apply simple transformations like adjusting brightness and contrast to the images. Here’s the implementation:

In [18]:
from typing import List, Dict, Tuple
import random
from PIL import Image, ImageEnhance

def apply_augmentations(img: Image.Image, annotations: List[Dict]) -> Tuple[List[Image.Image], List[List[Dict]]]:
    """
    Apply a single random augmentation to an image and generate corresponding annotations.

    Args:
        img (Image.Image): Input image.
        annotations (List[Dict]): List of COCO annotations for the image.

    Returns:
        Tuple[List[Image.Image], List[List[Dict]]]: A tuple containing a list with one augmented image
                                                   and a list with the corresponding annotations.
    """
    # Define augmentations and their ranges
    augmentations = [
        (ImageEnhance.Brightness, (0.8, 1.2)),  # Brightness adjustment
        (ImageEnhance.Contrast, (0.8, 1.2)),    # Contrast adjustment
    ]

    # Randomly choose an augmentation to apply
    augmentation, factor_range = random.choice(augmentations)
    enhancer = augmentation(img)
    augmented_img = enhancer.enhance(random.uniform(*factor_range))

    # Return a single augmented image and its annotations
    return [augmented_img], [deepcopy(annotations)]


The `process_dataset` function contains the main logic for processing the dataset. It performs the following steps:
1. Applies the defined augmentations to all images in the input dataset.
2. Saves the augmented images to the output dataset.
3. Copies the annotations from the input dataset to the output dataset unchanged, as the augmentations do not modify the annotation structure.

In [19]:
from examples.processing.augmentation.utils.augmentations import get_annotations_for_image, \
    save_augmented_images_and_update_coco
from copy import deepcopy
from typing import Dict
import os
from src import step
from glob import glob

from src.models.dataset.common.coco_dataset_context import CocoDatasetContext

@step
def process_dataset(input_dataset: CocoDatasetContext, output_dataset: CocoDatasetContext):
    """
    Apply augmentations to the input dataset images and save the augmented images in the output dataset.

    Args:
        input_dataset (object): Input dataset object with attributes like `images_dir` and `coco_data`.
        output_dataset (object): Output dataset object where augmented images and annotations are stored.
    """
    # Initialize COCO structure for the output dataset
    output_coco_data: Dict = deepcopy(input_dataset.coco_data)
    output_coco_data["images"] = []
    output_coco_data["annotations"] = []

    # List all images in the input dataset
    image_paths = glob(os.path.join(input_dataset.images_dir, "*"))

    for img_id, image_path in enumerate(image_paths):
        # Open the image
        img = Image.open(image_path)

        # Get associated annotations
        annotations = get_annotations_for_image(coco_data=input_dataset.coco_data, image_id=img_id)

        # Apply augmentations
        augmented_images, augmented_annotations = apply_augmentations(img, annotations)

        # Save augmented images and update COCO metadata
        save_augmented_images_and_update_coco(
            augmented_images=augmented_images,
            augmented_annotations=augmented_annotations,
            output_coco_data=output_coco_data,
            output_dir=output_dataset.images_dir,
            base_filename=os.path.basename(image_path)
        )

    # Save updated COCO data
    output_dataset.coco_data = output_coco_data

    print(f"Processed {len(image_paths)} images and saved augmented images to {output_dataset.images_dir}.")

    return output_dataset

ValueError: More than one step is called 'process_dataset'. The step names must be unique.

Now, let’s execute the processing pipeline by calling the `process_dataset` function. This will:
- Apply the augmentations to all images in the input dataset.
- Save the augmented images to the output dataset.
- Copy the annotations to the output dataset.

In [20]:
@pipeline(context=local_processing_context, log_folder_path="logs/", remove_logs_on_completion=False)
def augmentations_pipeline():
    dataset_collection = get_processing_dataset_collection()
    dataset_collection["output"] = process_dataset(dataset_collection["input"], dataset_collection["output"])
    return dataset_collection

Run the pipeline to process the datasets and store the augmented data:


In [21]:
dataset_collection = augmentations_pipeline()

Inspect the directory where the augmented images are saved:


In [22]:
dataset_collection["output"].images_dir

'/home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/examples/processing/augmentation/test_augmentations/images/output'

Check the path to the updated COCO annotation file for the augmented dataset:

In [23]:
dataset_collection["output"].coco_file_path

'/home/sonia/Documents/Picsellia/picsellia_repos/picsellia-training-engine/examples/processing/augmentation/test_augmentations/annotations/output/annotations.json'

Once the processing is complete, the output dataset is ready to be uploaded to Picsellia. This ensures that the results are available for experimentation or future use. Here’s how to upload it:

In [24]:
from src.steps.processing.common.dataset_context_uploader import upload_dataset_context

@pipeline(context=local_processing_context, log_folder_path="logs/", remove_logs_on_completion=False)
def augmentations_pipeline():
    dataset_collection = get_processing_dataset_collection()
    dataset_collection["output"] = process_dataset(dataset_collection["input"], dataset_collection["output"])
    upload_dataset_context(dataset_collection["output"], use_id=False)

Now if you run it with the `augmentations_pipeline` function, the input dataset will be processed and the output dataset will be uploaded to Picsellia. Here, no need to create a processing on picsellia because the context given is local.

In [25]:
augmentations_pipeline()

If you want to create a Picsellia processing, you simply need to change the given context to a Picsellia context and run the pipeline. The pipeline would look like this:
If you want the following cell it will not run because this pipeline can only be run on Picsellia.

In [26]:
# @pipeline(context=create_picsellia_processing_context(parameters), log_folder_path="logs/", remove_logs_on_completion=False)
# def augmentations_pipeline():
#     dataset_collection = get_processing_dataset_collection()
#     dataset_collection["output"] = process_dataset(dataset_collection["input"], dataset_collection["output"])
#     upload_dataset_context(dataset_collection["output"], use_id=False)

To make this pipeline run, we need to dockerize it and upload it to Picsellia.

In [27]:
from src.models.utils.processing_creation import setup_dockerized_pipeline

setup_dockerized_pipeline(
    api_token="",
    organization_id="",
    processing_name="augmentations_pipeline",
    pipeline_script_path="augmentations_pipeline.py",
    requirements_file_path="requirements.txt",
    docker_image="soniagrh/processing-augmentations",
    docker_tag="latest",
    default_parameters={
        "augmentation_probability": 0.5,
        "datalake": "default",
        "data_tag": "augmented_data",
    },
    base_docker_image="picsellia/cpu:python3.10"
)